# Two-qubit chip: eigenmode and S-parameters

Build a filled QPDK chip, find an eigenmode, and sweep S11 and S21 using **one shared mesh**.
The [two-qubit layout](https://github.com/gdsfactory/gsim/blob/main/nbs/data/qpdk/two_qubit_straight_bus.pic.yml)
is a straight-bus variant of QPDK's filled test chip.

**Requirements:**

- QPDK: `uv pip install qpdk==0.4.0`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

The model uses planar PEC metal on silicon with vacuum above. Junctions are
omitted, and both launcher ports have matched 50 Ω loads, so the eigenmode Q
includes port loading. Lengths are in µm; frequencies below are in GHz.

In [ ]:
from pathlib import Path

import gdsfactory as gf
import klayout.db as kdb
import pandas as pd
from qpdk import PDK
from qpdk.cells.chip import chip_edge
from qpdk.tech import (
    LAYER,
    LAYER_STACK,
    get_etch_section,
    material_properties,
    routing_strategies,
)
from qpdk.utils import apply_additive_metals, fill_magnetic_vortices

from gsim import gcloud
from gsim.common.polygon_utils import simplify_polygon
from gsim.common.stack import Layer, LayerStack
from gsim.palace import DrivenConfig, EigenmodeConfig, SolverConfig
from gsim.palace.mesh import generate_mesh, write_config
from gsim.palace.ports import configure_cpw_port, extract_ports
from gsim.viz import plot_mesh

PDK.activate()

### Settings

Start with the current 8 µm mesh for an overview. The layout contains 2 µm
features; compare finer meshes before relying on resonance frequencies or Q.
Each refinement writes to a separate directory.

In [ ]:
refined_mesh_size_um = 8.0
max_mesh_size_um = 300.0
curve_tolerance_um = 0.1
chip_padding_um = 50.0
chip_edge_width_um = 100.0
port_length_um = 10.0

eigenmode_target_ghz = 5.0
num_eigenmodes = 1
driven_fmin_ghz = 7.0
driven_fmax_ghz = 10.0
driven_num_points = 10001
driven_adaptive_tol = 1e-3
driven_adaptive_max_samples = 40

notebook_dir = Path("nbs") if Path("nbs").is_dir() else Path.cwd()
case_dir = Path(f"palace-sim-qpdk-two-qubit-{refined_mesh_size_um:g}um")
input_dir = case_dir / "inputs"
case_dir.mkdir(parents=True, exist_ok=True)

### Build the chip

Add vortex-trapping fill and leave 50 µm of ground between the layout,
including the launchers, and the 100 µm-wide etched chip-edge frame.

In [ ]:
port_layout = gf.read.from_yaml(
    notebook_dir / "data/qpdk/two_qubit_straight_bus.pic.yml",
    routing_strategies=routing_strategies,
)
assert chip_padding_um > 0
assert chip_edge_width_um > 0
chip_bounds = port_layout.dbbox().enlarged(chip_padding_um + chip_edge_width_um)
filled_layout = gf.Component()
filled_layout << fill_magnetic_vortices(
    component=port_layout,
    rectangle_size=(15.0, 15.0),
    gap=70.0,
    stagger=2,
)
edge = filled_layout << chip_edge(
    size=(chip_bounds.width(), chip_bounds.height()),
    width=chip_edge_width_um,
    layer=LAYER.M1_ETCH,
)
edge.dmove((chip_bounds.left, chip_bounds.bottom))
chip = apply_additive_metals(filled_layout)
chip.write_gds(case_dir / "layout.gds")
chip.plot()

### Convert the etch mask to metal

Subtract the etch mask from the chip footprint. Simplify curved boundaries
while checking that the number of metal islands and holes is preserved.
Copy the two bus ports from the launcher pads.

In [ ]:
layout = chip.kdb_cell.layout()
etch = kdb.Region(chip.kdb_cell.begin_shapes_rec(layout.layer(*LAYER.M1_ETCH))).merged()
assert not etch.is_empty(), "Missing M1 etch mask."
footprint = kdb.Region(etch.bbox())
original_metal = footprint - etch
if curve_tolerance_um > 0:
    etch = kdb.Region(
        [simplify_polygon(p, tolerance=curve_tolerance_um) for p in etch.each()]
    ).merged()
metal = footprint - etch
assert metal.count() == original_metal.count()
assert sum(p.holes() for p in metal.each()) == sum(
    p.holes() for p in original_metal.each()
)

mesh_layers = {"M1": (201, 0), "SUBSTRATE": (202, 0), "VACUUM": (203, 0)}
mesh_component = gf.Component()
for name, region in (("M1", metal), ("SUBSTRATE", footprint), ("VACUUM", footprint)):
    mesh_component.kdb_cell.shapes(
        mesh_component.kcl.layout.layer(*mesh_layers[name])
    ).insert(region)

# Keep the exported ports on the wide launcher pads.
for port_name in ("o1", "o2"):
    mesh_component.add_port(name=port_name, port=port_layout.ports[port_name])

### Define the stack and ports

Use QPDK's silicon and vacuum thicknesses, with the PEC sheet at `z=0`.
The CPW ports span the launcher gaps and remain inside the signal pads.

In [ ]:
stack = LayerStack(pdk_name="qpdk")
for name, zmin, zmax, material, layer_type in (
    (
        "SUBSTRATE",
        -LAYER_STACK.layers["Substrate"].thickness,
        0.0,
        "silicon",
        "dielectric",
    ),
    ("M1", 0.0, 0.0, "niobium", "conductor"),
    ("VACUUM", 0.0, LAYER_STACK.layers["Vacuum"].thickness, "vacuum", "dielectric"),
):
    stack.layers[name] = Layer(
        name=name,
        gds_layer=mesh_layers[name],
        zmin=zmin,
        zmax=zmax,
        thickness=zmax - zmin,
        material=material,
        layer_type=layer_type,
    )
stack.dielectrics = [
    dict(
        name=layer.name.lower(),
        zmin=layer.zmin,
        zmax=layer.zmax,
        material=layer.material,
    )
    for layer in stack.layers.values()
    if layer.layer_type == "dielectric"
]
stack.materials = {
    "silicon": {
        "type": "dielectric",
        "permittivity": material_properties["Si"]["relative_permittivity"],
        "loss_tangent": material_properties["Si"]["loss_tangent"],
    },
    "vacuum": {"type": "dielectric", "permittivity": 1.0},
    "niobium": {"type": "conductor"},
}

In [ ]:
for port_name in ("o1", "o2"):
    port = mesh_component.ports[port_name]
    configure_cpw_port(
        port,
        layer="M1",
        s_width=float(port.dwidth),
        gap_width=get_etch_section(port.info["cross_section"]).width,
        length=port_length_um,
        offset=port_length_um / 2,
        impedance=50.0,
        excited=False,
    )
ports = extract_ports(mesh_component, stack)
solver = SolverConfig(linear={"tolerance": 1e-9})
eigenmode_settings = EigenmodeConfig(
    num_modes=num_eigenmodes, target=eigenmode_target_ghz * 1e9
)

### Generate the shared mesh

Embed both port surfaces before meshing. The eigenmode and driven
configurations below reuse this mesh; changing geometry or port positions
requires rerunning this section. Inspect `P1` and `P2` before submitting jobs.

In [ ]:
mesh = generate_mesh(
    component=mesh_component,
    stack=stack,
    ports=ports,
    output_dir=input_dir,
    refined_mesh_size=refined_mesh_size_um,
    max_mesh_size=max_mesh_size_um,
    margin_x=0.0,
    margin_y=0.0,
    air_margin=0.0,
    fmax=driven_fmax_ghz * 1e9,
    planar_conductors=True,
    simulation_type="eigenmode",
    eigenmode_config=eigenmode_settings,
    numerical_config=solver,
    write_config=False,
    verbosity=0,
)
assert mesh.mesh_stats["sicn"]["invalid"] == 0
assert set(mesh.groups["port_surfaces"]) == {"P1", "P2"}

In [ ]:
plot_mesh(
    mesh.mesh_path,
    output=case_dir / "mesh_ports.png",
    show_groups=["M1", "P1", "P2"],
    interactive=False,
)

### Run the eigenmode simulation

Write a configuration for the shared mesh and run it on GDSFactory+ cloud.
The helper reuses matching completed jobs. Downloads stay outside the input
folder so they do not enter subsequent uploads.

In [ ]:
def run_cloud_job(inputs: Path, results_dir: Path):
    """Submit the current configuration or reuse matching completed results."""
    input_hash, job_id = gcloud.check_cache_for_dir(inputs, "palace")
    if job_id is None:
        job_id = gcloud.upload(inputs, "palace", input_hash=input_hash)
        gcloud.start(job_id)
    print(f"Cloud job: {job_id}")
    return gcloud.wait_for_results(job_id, parent_dir=results_dir)

In [ ]:
write_config(
    mesh,
    stack,
    ports,
    simulation_type="eigenmode",
    eigenmode_config=eigenmode_settings,
    numerical_config=solver,
)
eigenmode_results = run_cloud_job(input_dir, case_dir / "eigenmode-results")

In [ ]:
eigenmodes = pd.read_csv(eigenmode_results["eig.csv"], skipinitialspace=True)
eigenmodes.columns = eigenmodes.columns.str.strip()
eigenmodes

### Run the driven simulation

Sweep 7–10 GHz, excite `o1`, and leave `o2` matched to 50 Ω. Only the solver
configuration and excitation change; the mesh is reused. The adaptive sweep
builds a reduced model and evaluates the requested output grid.

A narrow resonance may need a smaller frequency span and finer sampling.
Zoom around a feature and compare results across frequency and mesh refinements.

In [ ]:
for port in ports:
    port.excited = port.name == "o1"

driven_settings = DrivenConfig(
    fmin=driven_fmin_ghz * 1e9,
    fmax=driven_fmax_ghz * 1e9,
    num_points=driven_num_points,
    adaptive_tol=driven_adaptive_tol,
    adaptive_max_samples=driven_adaptive_max_samples,
    reference_impedance=50.0,
    excitation_port="o1",
)
write_config(
    mesh,
    stack,
    ports,
    simulation_type="driven",
    driven_config=driven_settings,
    numerical_config=solver,
)
frequency_step_khz = (driven_fmax_ghz - driven_fmin_ghz) * 1e6 / (driven_num_points - 1)
print(f"Output frequency spacing: {frequency_step_khz:g} kHz")

In [ ]:
driven_sparams = run_cloud_job(input_dir, case_dir / "driven-results")

### Plot S-parameters

Plot S11 and S21 magnitude and phase. Dotted lines mark eigenfrequencies
within the driven sweep as references for the port-loaded response.

In [ ]:
sparam_figure = driven_sparams.plot_plotly()
for frequency_ghz in eigenmodes["Re{f} (GHz)"]:
    if driven_fmin_ghz <= frequency_ghz <= driven_fmax_ghz:
        sparam_figure.add_vline(
            x=float(frequency_ghz),
            line_dash="dot",
            line_color="gray",
            row="all",
            col=1,
        )
sparam_figure.update_layout(title="Two-qubit chip: S11 and S21")
sparam_figure